# CW2 Scaffold B: Tree-Based Factor Investing

Do non-linear models outperform linear benchmarks for UK equity factor return prediction?

**Module:** FIN306  
**Instructions:** Run provided cells to produce baseline results. Complete the `# TODO` sections. Write a 2,500-word reflective report addressing the five components in the CW2 brief.

---

## What you will do

| Step | Task | Status |
|------|------|--------|
| 1 | Load JKP UK factor data | Provided |
| 2 | Explore factor returns and Sharpe ratios | Provided |
| 3 | Walk-forward OLS baseline | Provided |
| 4 | Walk-forward Ridge regression | Provided |
| 5 | Add decision tree and random forest | **TODO 1** |
| 6 | Bias-variance tradeoff plot | **TODO 2** |
| 7 | SHAP feature importance | **TODO 3** |
| 8 | Economic significance: compare to naive benchmark | **TODO 4** |
| 9 | Extension: does adding momentum improve all models? | **TODO 5 (optional)** |

---

## Key concepts

**Walk-forward (expanding-window) evaluation** is the correct way to assess predictive models on time-series data. At each step $t$, you train on all data up to $t-1$ and predict at $t$ — no future information enters the model. Out-of-sample $R^2$ is the honest metric:
$$R^2_{OOS} = 1 - \frac{\sum_t (y_t - \hat{y}_t)^2}{\sum_t (y_t - \bar{y}_{train})^2}$$

Negative $R^2_{OOS}$ means the model performs worse than predicting the historical mean. This is the typical finding for monthly equity return prediction.

**Ridge regression** adds an $\ell_2$ penalty to OLS: $\hat{\beta}_{ridge} = (X^\top X + \lambda I)^{-1} X^\top y$. The regularisation parameter $\lambda$ is chosen by time-series cross-validation. Ridge shrinks coefficients toward zero, reducing variance at the cost of some bias.

**Decision trees and random forests** partition the feature space non-linearly. Deep trees have low bias but high variance (overfit). Shallow trees have high bias but low variance. The bias-variance tradeoff is visible as a function of tree depth. Random forests average many decorrelated trees to reduce variance without increasing bias as sharply.

**SHAP (SHapley Additive exPlanations)** decomposes each model prediction into factor contributions, enabling consistent comparison of feature importance across models.

## Before you code: state your research claim

Good data science starts with a question that evidence can refute. Before running any cells, read the claim below and ask yourself the four questions that follow.

---

**Starting claim:**

> *Tree-based models (decision tree, random forest) do not materially improve on Ridge regression for out-of-sample prediction of UK equity factor returns at monthly frequency. The bias-variance tradeoff favours simpler models in short, noisy financial time series: the additional flexibility of tree-based methods introduces variance that cannot be recovered with the available data.*

---

**Think through these questions before you start:**

1. **Why might tree-based models fail to outperform Ridge on monthly returns?** How many monthly observations does 20 years of UK factor data give you? Is that enough to fit a model with dozens of non-linear interactions?

2. **What does a negative $R^2_{OOS}$ mean?** If all models produce negative out-of-sample $R^2$, does that mean the exercise is pointless? What would a practitioner learn from the ranking of the models, even if none beats the naive mean?

3. **What would falsify this claim?** If random forest walk-forward $R^2$ is materially better than Ridge, what would that tell you about non-linearities in UK factor premia? Which factor relationships might be non-linear?

4. **What is the difference between statistical significance and economic significance?** A model might have a higher $R^2_{OOS}$ than Ridge by 0.001. Would that be worth using in a real portfolio? Think about transaction costs, capacity constraints, and the cost of complexity.

---

Write a one- or two-sentence version of *your own* refined claim in the cell below — before running any code. Your report introduction should open with this claim.

**Your refined claim (write here before running any code):**

*Replace this text with your own one- or two-sentence falsifiable claim. You will return to this cell after completing the analysis and assess whether your results support or refute it.*

## Setup

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
import statsmodels.api as sm

# SHAP for feature importance
try:
    import shap
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap

print('Setup complete.')

## Step 1: Load JKP UK factor data

In [ ]:
# Data: Jensen, Kelly & Pedersen (2024) global factor dataset, UK subset
DATA_URL = 'https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/main/resources/jkp_master_global_monthly.csv'
DATA_LOCAL = Path('labs/jkp_master_global_monthly.csv')

for path in [DATA_LOCAL, Path('../labs/jkp_master_global_monthly.csv'), Path('jkp_master_global_monthly.csv')]:
    if path.exists():
        df_raw = pd.read_csv(path)
        print(f'Loaded from local: {path}')
        break
else:
    print('Downloading from GitHub...')
    df_raw = pd.read_csv(DATA_URL)
    print('Download complete.')

df_raw['date'] = pd.to_datetime(df_raw['date'])
uk = df_raw[df_raw['country'] == 'gbr'].copy().sort_values('date').set_index('date')

factors = ['HML', 'SMB', 'MOM', 'RMW', 'CMA']
factors = [f for f in factors if f in uk.columns]
uk_factors = uk[factors].dropna()

print(f'\nUK factors loaded: {factors}')
print(f'Period: {uk_factors.index[0].date()} to {uk_factors.index[-1].date()}')
print(f'Observations: {len(uk_factors)}')
uk_factors.describe().round(4)

## Step 2: Explore factor returns

In [ ]:
# Cumulative factor returns
cumulative = (1 + uk_factors).cumprod()

fig, ax = plt.subplots(figsize=(11, 5))
for col in factors:
    ax.plot(cumulative.index, cumulative[col], label=col, linewidth=1.5)
ax.axhline(1, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Cumulative UK factor returns (JKP dataset)')
ax.set_ylabel('Cumulative return (£1 invested)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def annualised_sharpe(returns):
    mean = returns.mean() * 12
    vol  = returns.std() * np.sqrt(12)
    return mean / vol if vol > 0 else np.nan

sharpe = uk_factors.apply(annualised_sharpe)
print('Annualised Sharpe ratios:')
print(sharpe.round(3).to_string())

# Equal-weighted combination of factors with positive long-run Sharpe
positive_factors = [f for f in factors if sharpe[f] > 0]
if not positive_factors:
    positive_factors = factors  # fallback: use all
uk_factors['EW'] = uk_factors[positive_factors].mean(axis=1)
print(f'\nEqual-weighted portfolio factors: {positive_factors}')
print(f'EW Sharpe: {annualised_sharpe(uk_factors["EW"]):.3f}')

## Step 3: Walk-forward OLS and Ridge baseline

We predict next month's equal-weighted factor return using lagged factor returns. The train/test split is set at 60% to leave a meaningful out-of-sample window.

In [ ]:
predictors = positive_factors
target     = 'EW'

# One-period lagged features: predict t using factors at t-1
X = uk_factors[predictors].shift(1).dropna()
y = uk_factors[target].loc[X.index]

split = int(len(X) * 0.60)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f'Training: {X_train.index[0].date()} to {X_train.index[-1].date()} ({len(X_train)} obs)')
print(f'Test:     {X_test.index[0].date()}  to {X_test.index[-1].date()}  ({len(X_test)} obs)')

# --- In-sample OLS ---
ols = LinearRegression().fit(X_train, y_train)
r2_ols_in  = ols.score(X_train, y_train)
r2_ols_out = r2_score(y_test, ols.predict(X_test))

# --- In-sample Ridge (λ chosen by TimeSeriesSplit CV) ---
alphas = np.logspace(-3, 3, 50)
ridge  = RidgeCV(alphas=alphas, cv=TimeSeriesSplit(n_splits=5))
ridge.fit(X_train, y_train)
r2_ridge_in  = ridge.score(X_train, y_train)
r2_ridge_out = r2_score(y_test, ridge.predict(X_test))

print(f'\n{"Model":<14} {"In-sample R²":>14} {"Out-of-sample R²":>18}')
print('-' * 50)
print(f'{"OLS":<14} {r2_ols_in:>14.4f} {r2_ols_out:>18.4f}')
print(f'{"Ridge (λ=" + f"{ridge.alpha_:.2f})"):<14} {r2_ridge_in:>14.4f} {r2_ridge_out:>18.4f}')
print()
print('Note: negative OOS R² means the model is outperformed by the historical mean.')
print('This is the typical finding for monthly equity return prediction.')

## Step 4: Walk-forward evaluation (expanding window)

Static train/test is informative but walk-forward is the honest standard. We grow the training window by one month at a time, always predicting the next period.

In [ ]:
min_train = 60  # minimum 5 years to initialise
preds_ols   = []
preds_ridge = []
actuals     = []

for t in range(min_train, len(X) - 1):
    X_tr, y_tr = X.iloc[:t], y.iloc[:t]
    X_next     = X.iloc[t:t+1]

    ols_wf = LinearRegression().fit(X_tr, y_tr)
    preds_ols.append(ols_wf.predict(X_next)[0])

    ridge_wf = RidgeCV(alphas=np.logspace(-3, 3, 30), cv=TimeSeriesSplit(n_splits=3))
    ridge_wf.fit(X_tr, y_tr)
    preds_ridge.append(ridge_wf.predict(X_next)[0])

    actuals.append(y.iloc[t])

actuals = np.array(actuals)
wf_r2_ols   = r2_score(actuals, preds_ols)
wf_r2_ridge = r2_score(actuals, preds_ridge)

print(f'Walk-forward OOS R²:')
print(f'  OLS:   {wf_r2_ols:.4f}')
print(f'  Ridge: {wf_r2_ridge:.4f}')

## Step 5: Add tree-based models

### TODO 1: Decision tree and random forest

**Your tasks:**

1. In the static evaluation (Step 3 pattern), fit a `DecisionTreeRegressor(max_depth=3)` and a `RandomForestRegressor(n_estimators=100, max_depth=3)` on the training set. Report in-sample and out-of-sample $R^2$.
2. Extend the walk-forward loop from Step 4 to also generate predictions from these two models. Store as `preds_tree` and `preds_rf`.
3. Report all four walk-forward $R^2$ values in a summary table.

**In your report:** do the tree-based models outperform the linear benchmarks out-of-sample? What does the direction of the result tell you about non-linear structure in UK factor returns at this sample size?

In [ ]:
# TODO 1 — your code here
# Static:
#   tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)
#   rf   = RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42).fit(X_train, y_train)

# Walk-forward: extend the loop above, adding .predict(X_next)[0] for each model



## Step 6: Bias-variance tradeoff

### TODO 2: Plot model performance as a function of tree depth

**Your tasks:**

1. Loop over `depths = [1, 2, 3, 5, 8, 12]`. For each depth, fit a `DecisionTreeRegressor(max_depth=depth)` on the training set and record both in-sample and out-of-sample $R^2$ (use the static X_train/X_test split for speed).
2. Plot both curves on a single chart: depth on the x-axis, $R^2$ on the y-axis. Use different colours or markers for in-sample vs out-of-sample.
3. Draw a horizontal dashed line at the Ridge out-of-sample $R^2$ (computed in Step 3).

**In your report:** at what depth does the in-sample/out-of-sample gap open up? What does this tell you about overfitting? Does any tree depth beat Ridge out-of-sample?

In [ ]:
# TODO 2 — your code here
# depths = [1, 2, 3, 5, 8, 12]
# for depth in depths:
#     tree = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_train, y_train)
#     r2_in  = tree.score(X_train, y_train)
#     r2_out = r2_score(y_test, tree.predict(X_test))



## Step 7: SHAP feature importance

### TODO 3: Explain which factors matter

SHAP values decompose each prediction into additive contributions from each feature, making feature importance comparable across models.

**Your tasks:**

1. Fit a `RandomForestRegressor(n_estimators=200, max_depth=3, random_state=42)` on the full training set.
2. Compute SHAP values using `shap.TreeExplainer`:
   ```python
   explainer   = shap.TreeExplainer(rf)
   shap_values = explainer.shap_values(X_test)
   ```
3. Create a SHAP summary bar plot: `shap.summary_plot(shap_values, X_test, plot_type='bar')`.
4. Identify the two most important factors. Do the same factors dominate the Ridge coefficients?

**In your report:** compare SHAP-based importance with Ridge coefficient magnitudes. If they differ, why might that be? (Think about multicollinearity — Ridge shrinks correlated coefficients together, while SHAP attributes importance independently.)

In [ ]:
# TODO 3 — your code here
# import shap  (imported in Setup)
# rf_shap = RandomForestRegressor(n_estimators=200, max_depth=3, random_state=42).fit(X_train, y_train)
# explainer   = shap.TreeExplainer(rf_shap)
# shap_values = explainer.shap_values(X_test)
# shap.summary_plot(shap_values, X_test, plot_type='bar')



## Step 8: Economic significance

### TODO 4: Compare to a naive benchmark

Statistical significance alone is not enough. The relevant benchmark for a practitioner is the **historical mean forecast** — simply predicting the average past return every period.

**Your tasks:**

1. Build an expanding-window naive forecast: at each step $t$, predict the mean of all returns seen so far. Store as `preds_naive`.
2. Compute walk-forward $R^2$ for the naive model. (It should equal exactly 0 by construction of the $R^2_{OOS}$ formula — confirm this.)
3. Convert walk-forward predictions to simulated portfolio returns: if the model predicts a positive return, invest £1; otherwise hold cash. Compare cumulative wealth across models.
4. Compute the annualised Sharpe ratio of each strategy. Is the difference between models economically meaningful once transaction costs (~25 bps per trade) are accounted for?

**In your report:** explain why low or negative $R^2_{OOS}$ does not necessarily mean a model has no value for a portfolio manager. What other criteria matter?

In [ ]:
# TODO 4 — your code here
# Naive expanding mean:
#   preds_naive = [y.iloc[:t].mean() for t in range(min_train, len(X)-1)]
#
# Portfolio Sharpe:
#   signal = np.array(preds_rf) > 0   # 1 = invest, 0 = cash
#   strat_returns = signal * actuals
#   sharpe = strat_returns.mean() / strat_returns.std() * np.sqrt(12)



## Step 9 (extension): Does adding momentum improve all models?

### TODO 5: Include MOM explicitly if not already in your factor set *(optional)*

The momentum factor (MOM) has a well-documented predictive relationship with future returns: past winners tend to continue winning over 1-12 month horizons. If MOM is not already in your `positive_factors` list (it may have been excluded because of a negative Sharpe), add it explicitly and re-run the walk-forward comparison.

**Your tasks:**

1. Re-define `predictors_with_mom = positive_factors + ['MOM']` (if MOM is not already included).
2. Re-run the walk-forward loop with `X_mom` (lagged values of all factors including MOM) for Ridge and random forest only.
3. Compare walk-forward $R^2$ with and without MOM.

**In your report:** does MOM improve prediction? Does the direction of the result match the academic literature on momentum? What would it imply if adding MOM hurts out-of-sample performance?

In [ ]:
# TODO 5 (optional extension) — your code here



---

## Reflective report prompts

Your 2,500-word report must address the five components from the CW2 brief. The prompts below are to help you draft each section.

### A. Method rationale and theoretical foundation (~500 words)

- Why do tree-based models represent a departure from the linear factor framework? What theoretical assumption do they relax?
- What is the bias-variance tradeoff, and why does it matter differently for monthly equity data (few observations, high noise) versus daily transaction data (many observations)?
- What is Ridge regression correcting for? Why is regularisation especially important when predictors are correlated (HML and CMA often are)?
- Why is walk-forward evaluation necessary for financial time-series data? What is wrong with randomly shuffling observations before splitting?

### B. Data quality and preparation decisions (~500 words)

- What is the JKP dataset? Who produced it and how are the factors constructed?
- What look-ahead bias risks exist in factor data? (Accounting data timing: annual reports are released months after the fiscal year end, but some databases use filing dates, others use fiscal year dates. Which does JKP use?)
- What is survivorship bias in factor data? How does the JKP global dataset handle it?
- Why is the UK factor data sparser than the US? What does that mean for the reliability of your estimates?

**Stronger submissions** will draw on the Lab 9 advanced extension (Part 5: "What Is Inside a Factor?"), where you build a momentum factor from individual S&P 500 stock prices and compare it to the JKP pre-built MOM factor. If you completed that exercise, discuss what you learned about how construction choices (universe, weighting, survivorship filtering) affect factor returns and why the JKP dataset's methodological assumptions matter for your results.

### C. Results interpretation and validation (~750 words)

- Report the walk-forward $R^2$ for all models. Which performs best out-of-sample?
- Is the best model's out-of-sample $R^2$ positive? If not, what does that mean?
- Compare SHAP and Ridge coefficient magnitudes. Which factors appear most important?
- Does the bias-variance plot show the expected pattern (in-sample R² increasing, OOS R² first rising then falling with depth)?
- Is the Sharpe ratio improvement economically meaningful after transaction costs?

### D. Limitations, assumptions, and potential pitfalls (~500 words)

- Why might lagged factor returns be insufficient predictors? What other predictors might help?
- How sensitive are your results to the min_train window (60 months)? Is 5 years enough to initialise a stable model?
- What is data snooping bias, and how might it affect your choice of tree depth or regularisation parameter? How does the time-series cross-validation for Ridge help?
- Why does SHAP attribution change across windows in an expanding walk-forward? Is this instability a problem for a real trading system?

### E. Appropriate use in FinTech contexts and ethics (~250 words)

- How would a robo-adviser or systematic fund use a model like this in practice?
- What FCA model risk management guidance applies to algorithmic trading strategies?
- What is the risk of overfitting in a live production system? How would you monitor for degrading model performance?
- Are there fairness or transparency obligations when automated models drive investment decisions?